# NB07: Per-Phylum Forest Plot

Log-odds ratios with 95% CIs for P×Metal, N×Metal, Phz×Metal across GTDB phyla.

Run equivalent script: `python src/08_forest_plot.py`

In [1]:
import pandas as pd
import os
DATA_DIR = os.path.join('..', 'data')
FIG_DIR = os.path.join('..', 'figures')

In [2]:
forest = pd.read_csv(os.path.join(DATA_DIR, 'forest_plot_data.csv'))
print(f'{len(forest)} phylum-pair combinations')
print(f'Phyla represented: {forest["phylum"].nunique()}')
print(f'Pairs: {forest["pair"].unique().tolist()}')

102 phylum-pair combinations
Phyla represented: 34
Pairs: ['P x Metal', 'N x Metal', 'Phz x Metal']


## Top effects per pair

In [3]:
for pair in ['P x Metal', 'N x Metal', 'Phz x Metal']:
    print(f'\n{pair} — top 5 phyla by log-OR:')
    sub = forest[forest['pair']==pair].sort_values('log_OR', ascending=False).head(5)
    for _, r in sub.iterrows():
        sig = '*' if r['fisher_p'] < 0.05 else ''
        print(f"  {r['phylum']:>30s}: log-OR={r['log_OR']:+.3f} "
              f"[{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}] p={r['fisher_p']:.2e} {sig}")


P x Metal — top 5 phyla by log-OR:
                 p__Deinococcota: log-OR=+4.654 [-0.155, +9.463] p=1.00e+00 
           p__Desulfobacterota_F: log-OR=+3.807 [-0.434, +8.047] p=1.00e+00 
                 p__Nitrospirota: log-OR=+3.470 [-0.592, +7.532] p=1.00e+00 
               p__Fusobacteriota: log-OR=+2.931 [+0.872, +4.990] p=9.64e-03 *
                  p__Bacillota_B: log-OR=+2.865 [-1.148, +6.878] p=1.00e+00 

N x Metal — top 5 phyla by log-OR:
            p__Methanobacteriota: log-OR=+5.004 [+0.198, +9.810] p=1.00e+00 
               p__Halobacteriota: log-OR=+3.835 [-0.175, +7.845] p=1.00e+00 
                p__Spirochaetota: log-OR=+2.724 [-0.106, +5.554] p=6.13e-03 *
                p__Nanoarchaeota: log-OR=+2.717 [+1.050, +4.385] p=8.16e-04 *
                p__Chloroflexota: log-OR=+2.568 [-0.242, +5.379] p=6.76e-03 *

Phz x Metal — top 5 phyla by log-OR:
               p__Pseudomonadota: log-OR=+1.865 [-0.933, +4.663] p=1.07e-01 
               p__Actinomycetota: log-O

## Forest Plot (Figure 2)

In [4]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pairs = ['P x Metal', 'N x Metal', 'Phz x Metal']

fig, axes = plt.subplots(1, 3, figsize=(18, 10), sharey=False)

for ax, pair_label in zip(axes, pairs):
    sub = forest[forest['pair'] == pair_label].copy()
    sub = sub.sort_values('log_OR', ascending=True)

    y_pos = np.arange(len(sub))
    colors = ['#2166ac' if r['fisher_p'] < 0.05 else '#999999' for _, r in sub.iterrows()]

    ax.barh(y_pos, sub['log_OR'].values,
            xerr=[sub['log_OR'].values - sub['ci_lo'].values,
                  sub['ci_hi'].values - sub['log_OR'].values],
            height=0.6, color=colors, edgecolor='none', alpha=0.8,
            error_kw=dict(ecolor='#333333', capsize=2, linewidth=0.8))

    ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(y_pos)

    ylabels = []
    for _, r in sub.iterrows():
        p = r['phylum'].replace('p__', '')
        ylabels.append(f"{p} (n={r['n_species']})")
    ax.set_yticklabels(ylabels, fontsize=7)

    ax.set_xlabel('log Odds Ratio', fontsize=10)
    ax.set_title(pair_label, fontsize=12, fontweight='bold')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'forest_plot.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved figures/forest_plot.png")

Saved figures/forest_plot.png
